# 3. Pemahaman Data (88 Fitur)
Memetakan seluruh kolom data mentah, lalu menelaah kualitas, kepercayaan, dan volatilitas yang tidak terlihat dari harga saja.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw_path = '../data/raw/IDN_RTFP_mkt_2007_2026-08-24.csv'
data_path = '../data/processed/data_bersih.csv'
output_dir = '../data/processed'

KOL_PASAR = 'mkt_name'
KOL_PROVINSI = 'adm1_name'
KOL_TANGGAL = 'date'
KOMODITAS = ['eggs', 'garlic', 'meat_beef', 'meat_chicken', 'meat_chicken_broiler', 'oil', 'onions', 'rice', 'sugar']
ITEM = KOMODITAS + ['food_price_index']

## Kamus variabel: kelompok kolom pada data mentah

In [ ]:
kolom_mentah = pd.read_csv(raw_path, nrows=0).columns.str.strip().tolist()
identitas = ['ISO3', 'country', 'adm1_name', 'adm2_name', 'mkt_name', 'lat', 'lon', 'geo_id']
waktu = ['price_date', 'year', 'month']
metadata = ['currency', 'components', 'start_dense_data', 'last_survey_point', 'data_coverage',
            'data_coverage_recent', 'index_confidence_score', 'spatially_interpolated']
kelompok = {
    'Identitas dan lokasi': identitas,
    'Waktu': waktu,
    'Metadata dan kualitas': metadata,
    'Harga mentah (survei langsung)': [k for k in KOMODITAS if k in kolom_mentah],
    'Harga open (o_)': [c for c in kolom_mentah if c.startswith('o_')],
    'Harga high (h_)': [c for c in kolom_mentah if c.startswith('h_')],
    'Harga low (l_)': [c for c in kolom_mentah if c.startswith('l_')],
    'Harga close (c_)': [c for c in kolom_mentah if c.startswith('c_')],
    'Inflasi tahunan (inflation_)': [c for c in kolom_mentah if c.startswith('inflation_')],
    'Skor kepercayaan (trust_)': [c for c in kolom_mentah if c.startswith('trust_')],
}
ringkas = pd.DataFrame({'jumlah_kolom': {k: len([c for c in v if c in kolom_mentah]) for k, v in kelompok.items()}})
print(f'Total kolom mentah: {len(kolom_mentah)} | terpetakan: {ringkas.jumlah_kolom.sum()}')
ringkas

In [ ]:
df = pd.read_csv(data_path, parse_dates=[KOL_TANGGAL])
tersisa = [c for c in kolom_mentah if c in df.columns or (c == 'price_date' and KOL_TANGGAL in df.columns)]
print(f'Kolom setelah cleaning: {df.shape[1]} (kolom satu nilai dibuang)')
print('Kolom dibuang:', [c for c in kolom_mentah if c not in df.columns and c != 'price_date'])

## Kepercayaan data (trust_) dan skor indeks

In [ ]:
trust = [f'trust_{k}' for k in ITEM if f'trust_{k}' in df.columns]
df[trust].describe().T[['mean', 'min', 'max']].round(2)

In [ ]:
df['tahun'] = df[KOL_TANGGAL].dt.year
tren_trust = df.groupby('tahun')[trust].mean().mean(axis=1)
tren_trust.plot(figsize=(10, 4), marker='o', title='Rata-rata skor kepercayaan per tahun')
plt.ylabel('Skor trust')
plt.grid(True)
plt.savefig(f'{output_dir}/grafik_trust_per_tahun.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
trust_komoditas = df[trust].mean().rename(lambda c: c.replace('trust_', '')).sort_values()
trust_komoditas.plot(kind='barh', figsize=(8, 5), title='Rata-rata skor kepercayaan per komoditas')
plt.show()

In [ ]:
kolom_meta = [c for c in ['index_confidence_score', 'data_coverage', 'data_coverage_recent'] if c in df.columns]
if kolom_meta:
    pasar = df.groupby(KOL_PASAR)[kolom_meta].mean()
    print(pasar.describe().round(2))
else:
    print('Skor indeks dan cakupan data bernilai tunggal sehingga sudah dibuang saat cleaning')

## Interpolasi spasial

In [ ]:
if 'spatially_interpolated' in df.columns:
    status_pasar = df.groupby(KOL_PASAR)['spatially_interpolated'].max()
    print(f'Pasar dengan harga hasil interpolasi: {int(status_pasar.sum())} dari {len(status_pasar)} ({status_pasar.mean() * 100:.1f}%)')
    gabung = df.groupby('spatially_interpolated')[trust].mean().mean(axis=1)
    print('Rata-rata trust: interpolasi = 0 vs 1')
    print(gabung.round(2).to_string())
else:
    print('Kolom spatially_interpolated bernilai tunggal dan sudah dibuang saat cleaning')

## Harga mentah vs harga estimasi
Harga mentah hanya terisi saat ada survei langsung; harga close (c_) adalah estimasi yang selalu tersedia.

In [ ]:
mentah = [k for k in KOMODITAS if k in df.columns]
terisi = df[mentah].notna().mean().mul(100).round(1).sort_values()
terisi.plot(kind='barh', figsize=(8, 5), title='Persen baris dengan observasi harga langsung')
plt.xlabel('Persen terisi')
plt.savefig(f'{output_dir}/grafik_observasi_langsung.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df[mentah].notna().groupby(df['tahun']).mean().mean(axis=1).mul(100).round(1).to_frame('persen_observasi_langsung')

## Volatilitas dalam bulan
Rentang harga = (high - low) / close; makin besar, makin tidak stabil harga dalam satu bulan.

In [ ]:
rentang = {}
for k in ITEM:
    h, l, c = f'h_{k}', f'l_{k}', f'c_{k}'
    if {h, l, c}.issubset(df.columns):
        rentang[k] = ((df[h] - df[l]) / df[c] * 100).mean()
rentang = pd.Series(rentang).sort_values()
rentang.plot(kind='barh', figsize=(8, 5), title='Rata-rata rentang harga dalam bulan (%)')
plt.savefig(f'{output_dir}/grafik_rentang_harga.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['rentang_indeks'] = (df['h_food_price_index'] - df['l_food_price_index']) / df['c_food_price_index'] * 100
df.groupby(KOL_PROVINSI)['rentang_indeks'].mean().sort_values(ascending=False).head(10).round(2).to_frame('rentang_rata_rata_persen')

## Hubungan kepercayaan dan volatilitas

In [ ]:
pasar_ringkas = df.groupby(KOL_PASAR).agg(trust=('trust_food_price_index', 'mean'), rentang=('rentang_indeks', 'mean'))
print('Korelasi trust dan rentang harga antar pasar:', round(pasar_ringkas.corr().iloc[0, 1], 3))